<a href="https://colab.research.google.com/github/Eliezer318/SPOT-dLLM/blob/main/colab/spot_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

# SPOT quickstart

**On the SPOT: Enhancing Speculative Decoding for Diffusion Language Models via Commit-Order Prediction**
· [code](https://github.com/Eliezer318/SPOT-dLLM) · [project page](https://eliezer318.github.io/SPOT-dLLM/) · [drafters on Hugging Face](https://huggingface.co/EliezerS/SPOT-dLLM)

A diffusion language model fills a block of masked positions over several passes. Its own predictions from the
previous pass already give the right token at almost every position, so the key is **which position it commits
next**. SPOT learns that order with a 0.62M-parameter drafter, drafts a tree of future block states,
and the model checks the whole tree in one pass.

This notebook runs **SDAR-1.7B-Chat-b32** with its SPOT drafter on a free Colab GPU and compares draft-free
decoding with SPOT, first on one prompt and then on a few GSM8K questions.

**Before you start:** *Runtime → Change runtime type → T4 GPU* (any GPU works; it also runs on CPU, slowly).

## 1. Install

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU found: the notebook still runs on CPU, slowly."
!git clone --depth 1 https://github.com/Eliezer318/SPOT-dLLM
%cd SPOT-dLLM
# transformers is pinned to 4.x: the vendored model code is written against it
!pip install -q "transformers==4.55.4" accelerate

## 2. Download the model

SDAR-1.7B-Chat-b32 comes from the Hugging Face hub (3.8 GB). The drafters ship with the repository in
`drafters/` and are mirrored on Hugging Face. GSM8K is used in section 5.

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download("JetLM/SDAR-1.7B-Chat-b32")
snapshot_download("openai/gsm8k", repo_type="dataset")

## 3. Load the model and its drafter

The T4 has no native bfloat16, so the model runs in float32 there (it fits in 16 GB); newer GPUs use bfloat16.

In [ ]:
import time
import torch
from spot.decode import DecodeConfig, generate
from spot.drafters import SlotDrafter
from spot.models import load_adapter

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" and torch.cuda.get_device_capability()[0] >= 8 else torch.float32
target = load_adapter("sdar17", device=device, dtype=dtype)
drafter = SlotDrafter("drafters/sdar1.7b_b32.pt", target)
GEN_LENGTH = 256   # tokens per answer, as in the paper
print(f"SDAR-1.7B-Chat-b32 on {device} in {str(dtype).replace('torch.', '')}, with its SPOT drafter")

## 4. One prompt: draft-free decoding vs SPOT

`budget` is the number of drafted block states the model checks per pass. `gamma` turns on **SPOT-γ**, which
drafts fewer states when the drafter is unsure, so each pass is cheaper. The first call warms up the GPU.

In [ ]:
def spot_decode(prompt, budget=0, gamma=0.0, temperature=0.0, seed=0, gen_length=None, pad=None):
    """One generation. budget=0 is draft-free decoding (one commit per pass); gamma>0 is SPOT-γ."""
    drafting = budget > 0
    if pad is None:  # SPOT pads every pass to the full budget; SPOT-γ does not
        pad = (0 if gamma > 0 else budget) if drafting else 0
    cfg = DecodeConfig(
        cache="on", unmasking="single", verify="match", suffix="skip", draft_layout="concat",
        draft="slot" if drafting else "none", slot_mode="ddtree_b", slot_tokgate=0.6,
        draft_budget=max(budget, 1), pad_pieces=pad, slot_gamma=gamma,
        gen_length=gen_length or GEN_LENGTH, block_length=32, temperature=temperature, seed=seed)
    messages = [{"role": "user", "content": prompt}] if isinstance(prompt, str) else prompt
    ids = target.encode_prompt(messages)
    return generate(target, ids, cfg, drafter=drafter if drafting else None)


question = ("Natalia sold clips to 48 of her friends in April, and then she sold half as many clips "
            "in May. How many clips did Natalia sell altogether in April and May?")
spot_decode(question, budget=8, gen_length=32)  # warm-up

runs = {
    "draft-free": spot_decode(question),
    "SPOT, budget 8": spot_decode(question, budget=8),
    "SPOT-γ, γ = 0.1": spot_decode(question, budget=8, gamma=0.1),
}
base = runs["draft-free"]["wall_s_net"]
print(f"{'':18}{'passes':>8}{'seconds':>10}{'speed-up':>10}")
for name, r in runs.items():
    print(f"{name:18}{r['nfe']:>8}{r['wall_s_net']:>10.2f}{base / r['wall_s_net']:>9.2f}x")
print("\n" + runs["SPOT, budget 8"]["text"])

### Is the output exact?

At T = 0, SPOT with a fixed budget writes token for token what draft-free decoding writes. GPU kernels round
slightly differently when the shape of a pass changes, so the check compares against a draft-free run padded to
the same shape as SPOT's passes.

In [ ]:
reference = spot_decode(question, budget=0, pad=8)  # draft-free, padded to SPOT's pass shape
print("SPOT output identical to draft-free decoding:", reference["text"] == runs["SPOT, budget 8"]["text"])

## 5. A small GSM8K benchmark

Ten GSM8K test questions, scored by exact match on the final number. Raise `n` for a tighter estimate; on a T4
each method takes a few seconds per question.

In [ ]:
from spot.tasks import load_task, score_task

n = 10
samples = load_task("gsm8k", limit=n)


def benchmark(**kw):
    total = {"accuracy": 0.0, "passes": 0.0, "seconds": 0.0}
    for s in samples:
        r = spot_decode(s.messages, **kw)
        total["accuracy"] += score_task("gsm8k", s, r["text"])
        total["passes"] += r["nfe"]
        total["seconds"] += r["wall_s_net"]
    return {k: v / len(samples) for k, v in total.items()}


results = {
    "draft-free": benchmark(),
    "SPOT, budget 8": benchmark(budget=8),
    "SPOT-γ, γ = 0.1": benchmark(budget=8, gamma=0.1),
}
base = results["draft-free"]["seconds"]
print(f"{'':18}{'accuracy':>9}{'passes':>8}{'seconds':>10}{'speed-up':>10}")
for name, m in results.items():
    print(f"{name:18}{m['accuracy']:>9.2f}{m['passes']:>8.1f}{m['seconds']:>10.2f}{base / m['seconds']:>9.2f}x")

## 6. Sampling at T = 1

When the model samples, the drafter and the model share the sampling noise (coupled-noise verification). A whole
tree can then be checked by equality, and the output still follows the model's own sampling distribution.

In [ ]:
for seed in range(3):
    r = spot_decode(question, budget=8, temperature=1.0, seed=seed)
    answer = " ".join(r["text"].split())
    print(f"seed {seed}: {r['nfe']} passes | ...{answer[-90:]}")

## 7. Next steps

* **Your own prompts:** call `spot_decode("...", budget=8)`.
* **The paper's main target:** on a GPU with at least 24 GB (an A100 or L4), load SDAR-8B instead. It downloads
  automatically (17 GB):

  ```python
  target = load_adapter("sdar", device="cuda", dtype=torch.bfloat16)
  drafter = SlotDrafter("drafters/sdar8b_b32.pt", target)
  ```
* **The benchmark harness** behind the paper's tables is `run_sweep.py`; the [README](https://github.com/Eliezer318/SPOT-dLLM#readme) lists every
  arm and the commands that reproduce the results.